# Financial Time Series Analysis — Personal Project

**Zunaira Haider** | MSc Mathematics in Science and Engineering | TU Munich  

---

This notebook started as a way to apply what I learned in **MA4800 Foundations of Data Analysis** 
and **CIT4230006 Causal Inference in Time Series** to something more concrete than problem sets.  

The idea was simple: take real stock price data, check whether it's actually clean and reliable, 
and then do some basic time series analysis on it. Turns out there's a lot more to "clean data" 
than I expected.

I'm using Apple (AAPL), Microsoft (MSFT), Google (GOOGL), Amazon (AMZN) and JPMorgan (JPM) 
as my test case — partly because they're well-known, partly because the data is freely available 
via `yfinance`.


## 0. Setup & Imports

First time running this I didn't have yfinance installed. `pip install yfinance` fixed it.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# yfinance for downloading stock data
try:
    import yfinance as yf
    print('yfinance loaded successfully')
except ImportError:
    print('yfinance not found — run: pip install yfinance')

# plot style — cleaner than default matplotlib
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. Loading the Data

I'm pulling 4 years of daily closing prices (2020–2024). I chose this window because it includes 
the COVID crash in early 2020, the recovery, and the 2022 rate hike period — so there's actually 
something interesting to analyse rather than just a flat upward trend.

I also added a simple caching mechanism so I don't re-download every time I re-run the notebook — 
learned that lesson after accidentally hitting the API 10 times in a row while debugging.


In [2]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM']
START   = '2020-01-01'
END     = '2024-01-01'

import os

def load_prices(tickers, start, end, cache='prices_cache.csv'):
    """
    Load from cache if available, otherwise download from yfinance.
    Saves to CSV so we don't hit the API every single run.
    """
    if os.path.exists(cache):
        print(f'Loading from cache: {cache}')
        df = pd.read_csv(cache, index_col=0, parse_dates=True)
        return df

    print('Downloading from yfinance...')
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    df = raw['Close']
    df.to_csv(cache)
    print(f'Saved to {cache}')
    return df

prices = load_prices(TICKERS, START, END)
print(f'\nShape: {prices.shape}')
print(f'Period: {prices.index[0].date()} to {prices.index[-1].date()}')
prices.head()

## 2. Data Quality Checks

This part took longer than I expected. I initially just jumped straight into analysis but then 
realised in the Foundations of Data Analysis lectures that you should never trust raw data.

I'm checking for:
- **Missing values** — trading days where no price was recorded
- **Outliers** — abnormally large daily moves (using Z-score, as covered in the time series course)
- **Zero or negative prices** — shouldn't exist but can happen with bad data feeds
- **Date gaps** — unexpectedly long gaps between consecutive rows
- **Duplicate dates** — same date appearing twice

In a real asset management context this kind of pipeline runs automatically every morning 
before the trading day starts.


In [3]:
# --- 2.1 Basic overview ---
print('=== BASIC INFO ===')
print(f'Rows     : {len(prices)}')
print(f'Columns  : {list(prices.columns)}')
print(f'\nDtypes:\n{prices.dtypes}')
print(f'\nFirst few rows:')
prices.head(3)

In [4]:
# --- 2.2 Missing values ---
missing = prices.isnull().sum()
missing_pct = (missing / len(prices) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
})

print('=== MISSING VALUES ===')
print(missing_df)

if missing.sum() == 0:
    print('\nNo missing values — good.')
else:
    print(f'\nTotal missing: {missing.sum()} — needs handling before analysis')

In [5]:
# --- 2.3 Outlier detection using Z-score ---
# I compute daily returns first, then flag anything beyond 3 std deviations.
# This is the standard approach — covered in the time series lecture.
# The threshold of 3 is a common choice; lower = more sensitive, higher = fewer flags.

returns = prices.pct_change().dropna()
THRESHOLD = 3.0

print(f'=== OUTLIERS (Z-score > {THRESHOLD}) ===')
total_outliers = 0

for ticker in returns.columns:
    col = returns[ticker].dropna()
    z_scores = np.abs(stats.zscore(col))
    outliers = col[z_scores > THRESHOLD]
    total_outliers += len(outliers)
    
    if len(outliers) > 0:
        print(f'\n{ticker}: {len(outliers)} outlier(s)')
        for date, val in outliers.items():
            print(f'   {date.date()}  →  {val*100:+.2f}% daily return')
    else:
        print(f'{ticker}: no outliers')

print(f'\nTotal flagged: {total_outliers}')
# Note: these aren't necessarily errors — a +7% day for AAPL might be
# a real earnings surprise. In a real pipeline you'd cross-check with
# a second data source before deciding whether to keep or remove them.

In [6]:
# --- 2.4 Zero / negative prices ---
# These should never occur for large-cap stocks.
# If they do, it's a data feed issue.

zeros     = (prices == 0).sum()
negatives = (prices < 0).sum()

print('=== ZERO & NEGATIVE PRICES ===')
print(f'Zero prices    : {zeros.sum()}')
print(f'Negative prices: {negatives.sum()}')

# --- 2.5 Date gaps ---
date_diffs = pd.Series(prices.index).diff().dropna()
large_gaps = date_diffs[date_diffs > pd.Timedelta(days=5)]

print(f'\n=== DATE GAPS (> 5 days) ===')
if len(large_gaps) == 0:
    print('No unusual gaps detected.')
else:
    for idx in large_gaps.index:
        d1 = prices.index[idx-1].date()
        d2 = prices.index[idx].date()
        print(f'  Gap: {d1} → {d2}  ({date_diffs.iloc[idx-1].days} days)')

# --- 2.6 Duplicate dates ---
dupes = prices.index.duplicated().sum()
print(f'\nDuplicate dates: {dupes}')

### Quality Check Summary

The data is mostly clean. The outliers flagged are likely genuine market events 
(e.g. large post-earnings moves) rather than data errors — in a real pipeline 
you'd cross-reference with a second data provider to confirm.  

No zeros, no negatives, no duplicate dates. Good to proceed.


## 3. Time Series Analysis

Now the actual interesting part. I'll compute returns and then look at 
rolling statistics, correlations, and drawdowns.


In [7]:
# Normalise prices to 100 at start date — makes visual comparison easier
# regardless of actual price level (MSFT trades at ~$240, JPM at ~$130 etc)

normed = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#003399', '#e63946', '#2a9d8f', '#e9c46a', '#f4a261']

for i, ticker in enumerate(normed.columns):
    ax.plot(normed.index, normed[ticker],
            label=ticker, color=colors[i], linewidth=1.8)

ax.set_title('Normalised Price Performance (Base = 100 at Jan 2020)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('outputs/01_normalised_prices.png', dpi=150)
plt.show()
print('The COVID crash in March 2020 is clearly visible for all tickers.')

In [8]:
# Daily returns — this is what we actually analyse statistically
# Log returns are more appropriate for modelling but % returns are more intuitive

daily_ret = prices.pct_change().dropna()
log_ret   = np.log(prices / prices.shift(1)).dropna()

print('Daily return statistics (%):')
print((daily_ret * 100).describe().round(3))

In [9]:
# Return distributions — checking for normality / fat tails
# In the time series course we discussed how financial returns often have
# heavier tails than a normal distribution (excess kurtosis > 0)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=False)

for i, ticker in enumerate(daily_ret.columns):
    ax  = axes[i]
    data = daily_ret[ticker] * 100
    
    ax.hist(data, bins=60, color=colors[i], alpha=0.7,
            edgecolor='white', density=True)
    
    # overlay normal distribution with same mean/std for comparison
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma),
            'k-', linewidth=1.5, label='Normal fit')
    
    kurt = data.kurtosis()
    ax.set_title(f'{ticker}\nkurtosis={kurt:.2f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Daily Return (%)')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Density')
fig.suptitle('Return Distributions vs Normal — Fat Tails Visible',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/02_return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Kurtosis > 0 for all tickers confirms fat tails — returns are NOT normally distributed.')
print('This is consistent with what we discussed in the time series lectures.')

In [10]:
# Rolling volatility — 30 day window, annualised
# Multiply daily std by sqrt(252) to annualise (252 trading days per year)
# Volatility clustering is clearly visible — calm periods followed by turbulent ones

roll_vol = daily_ret.rolling(30).std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(12, 4))
for i, ticker in enumerate(roll_vol.columns):
    ax.plot(roll_vol.index, roll_vol[ticker],
            label=ticker, color=colors[i], linewidth=1.4, alpha=0.85)

# mark the COVID spike
ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-01'),
           alpha=0.15, color='red', label='COVID crash')

ax.set_title('30-Day Rolling Annualised Volatility (%)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Volatility (%)')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('outputs/03_rolling_volatility.png', dpi=150)
plt.show()
print('Volatility clustering clearly visible — especially the COVID spike in March 2020.')

In [11]:
# Correlation matrix
# I was surprised how correlated these stocks are — expected more independence
# The high correlations make sense though: they're all large US tech/finance stocks
# that react to the same macro factors (interest rates, inflation etc)

corr = daily_ret.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, annot_kws={'size': 12},
            ax=ax, square=True)
ax.set_title('Pearson Correlation — Daily Returns',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('outputs/04_correlation_heatmap.png', dpi=150)
plt.show()

print('Highest correlation:', corr.stack().drop_duplicates().nlargest(3))

In [12]:
# Drawdown analysis — how far did each stock fall from its peak?
# This is one of the most important risk metrics in portfolio management

def compute_drawdown(price_series):
    peak = price_series.cummax()
    dd   = (price_series - peak) / peak * 100
    return dd

fig, ax = plt.subplots(figsize=(12, 5))
for i, ticker in enumerate(prices.columns):
    dd = compute_drawdown(prices[ticker])
    ax.fill_between(dd.index, dd, 0,
                    alpha=0.35, color=colors[i], label=ticker)
    ax.plot(dd.index, dd, color=colors[i], linewidth=1)
    
    # annotate max drawdown
    max_dd = dd.min()
    max_dd_date = dd.idxmin()
    ax.annotate(f'{max_dd:.1f}%',
                xy=(max_dd_date, max_dd),
                fontsize=8, color=colors[i],
                xytext=(10, -10), textcoords='offset points')

ax.set_title('Drawdown from Peak (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('outputs/05_drawdowns.png', dpi=150)
plt.show()

In [13]:
# Moving averages for AAPL — classic technical analysis
# MA20 = short term trend, MA50 = medium, MA200 = long term
# When MA20 crosses above MA200 it's called a 'golden cross' — bullish signal

ticker = 'AAPL'
s = prices[ticker]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(s.index, s,              label='Price',  color='#003399', lw=1.8, alpha=0.9)
ax.plot(s.index, s.rolling(20).mean(),  label='MA 20',  color='#e63946', lw=1.2, ls='--')
ax.plot(s.index, s.rolling(50).mean(),  label='MA 50',  color='#2a9d8f', lw=1.2, ls='--')
ax.plot(s.index, s.rolling(200).mean(), label='MA 200', color='#f4a261', lw=1.5, ls='-.')

ax.set_title(f'{ticker} — Price with Moving Averages',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('outputs/06_moving_averages_AAPL.png', dpi=150)
plt.show()

## 4. Performance Summary

Pulling together the key metrics for each stock.

**Sharpe ratio** = (annualised return − risk free rate) / annualised volatility  
I'm using 2% as the risk-free rate (approximate US T-bill rate for this period).  
A Sharpe > 1 is generally considered good.


In [14]:
TRADING_DAYS = 252
RF = 0.02  # risk-free rate

summary_rows = []
for ticker in prices.columns:
    r        = daily_ret[ticker].dropna()
    total_r  = (prices[ticker].iloc[-1] / prices[ticker].iloc[0] - 1) * 100
    ann_r    = ((1 + r.mean()) ** TRADING_DAYS - 1) * 100
    ann_vol  = r.std() * np.sqrt(TRADING_DAYS) * 100
    sharpe   = (ann_r/100 - RF) / (ann_vol/100)
    max_dd   = compute_drawdown(prices[ticker]).min()
    
    summary_rows.append({
        'Ticker'              : ticker,
        'Total Return (%)'    : round(total_r, 1),
        'Ann. Return (%)'     : round(ann_r, 1),
        'Ann. Volatility (%)' : round(ann_vol, 1),
        'Sharpe Ratio'        : round(sharpe, 2),
        'Max Drawdown (%)'    : round(max_dd, 1),
    })

summary = pd.DataFrame(summary_rows).set_index('Ticker')
print(summary.to_string())
summary.to_csv('outputs/performance_summary.csv')

In [15]:
# Bar charts for the 3 key metrics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, color, title) in zip(axes, [
    ('Ann. Return (%)',      '#003399', 'Annualised Return (%)'),
    ('Ann. Volatility (%)', '#e63946', 'Annualised Volatility (%)'),
    ('Sharpe Ratio',        '#2a9d8f', 'Sharpe Ratio'),
]):
    vals = summary[col]
    ax.bar(vals.index, vals, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticklabels(vals.index, rotation=30)

fig.suptitle('Performance Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/07_performance_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. What I Learned / Takeaways

A few things that surprised me or that I want to remember:

**On data quality:**  
Even 'clean' data from a reputable source like Yahoo Finance has outliers that need investigation. 
In a real environment you'd always cross-check flagged values against a second data provider 
before deciding to remove them. The automated pipeline structure makes this easy to extend.

**On returns:**  
The kurtosis values confirm what we covered in the time series course — financial returns have 
fat tails. This means models that assume normality (like basic mean-variance portfolio theory) 
underestimate the probability of extreme losses. Worth keeping in mind.

**On correlation:**  
I expected more independence between these stocks. The high correlations (~0.7 between the tech 
names) mean that holding all five doesn't give as much diversification as it looks like on paper.

**On volatility:**  
Volatility clustering is very clear in the rolling charts — calm periods followed by turbulent 
ones. This is the basis for GARCH models which I want to look at next.

**Next steps I want to explore:**  
- Try a GARCH(1,1) model on one of the return series (extending what I learned in CIT4230006)  
- Add a simple Granger causality test between the stocks  
- Try to replicate the quality check pipeline with a real API that has deliberate errors injected


---
*Zunaira Haider — TU Munich — May 2026*  
*github.com/ZunairaHaider*
